<a href="https://colab.research.google.com/github/HLZHarry/LLM-Practice/blob/main/ch03/Ch03_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)

# Create a pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=50,
    do_sample=False,
)

In [ ]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."

output = generator(prompt)

print(output[0]['generated_text'])

In [ ]:
print(model)

## How to Read the Phi-3 Model Architecture

Think of it as a **pipeline of layers** that transforms text tokens into predicted next tokens.

---

## High-Level Structure
```
Input Tokens
     ↓
embed_tokens       ← convert token IDs to vectors
     ↓
layers (x32)       ← the "thinking" happens here, repeated 32 times
     ↓
norm               ← final cleanup
     ↓
lm_head            ← convert back to vocabulary probabilities
     ↓
Output Token Probabilities
```

---

## Layer by Layer

### 1. `embed_tokens`: Embedding(32064, 3072)
- Converts each token ID into a **vector of size 3072**
- `32064` = total vocabulary size (number of known tokens)
- `3072` = the model's **hidden size** (every layer works with vectors of this size)

---

### 2. `layers`: 32 x `Phi3DecoderLayer`
The core of the model, repeated **32 times**. Each layer has:

#### `self_attn` — Attention (how tokens relate to each other)
- `qkv_proj`: Linear(3072 → 9216)
  - Projects input into **Q, K, V** matrices (9216 = 3072 × 3, one for each)
- `o_proj`: Linear(3072 → 3072)
  - Projects attention output back to hidden size

#### `mlp` — Feed Forward Network (per-token processing)
- `gate_up_proj`: Linear(3072 → 16384)
  - Expands to a larger space for richer computation
- `down_proj`: Linear(8192 → 3072)
  - Compresses back to hidden size
- `activation_fn`: SiLU — a smooth activation function (like ReLU but smoother)

#### Normalization & Dropout
- `input_layernorm` / `post_attention_layernorm`: **RMSNorm** — stabilizes values before/after attention
- `resid_attn_dropout` / `resid_mlp_dropout`: **Dropout(p=0.0)** — disabled during inference (only used in training)

---

### 3. `norm`: Phi3RMSNorm(3072)
- Final normalization after all 32 layers

---

### 4. `lm_head`: Linear(3072 → 32064)
- Projects the final hidden vector back to **vocabulary size**
- Produces a probability score for each of the 32064 possible next tokens
- The token with the highest score is selected as the output

---

## Key Numbers Summary

| Parameter | Value | Meaning |
|---|---|---|
| Vocabulary size | 32064 | Number of known tokens |
| Hidden size | 3072 | Vector size throughout the model |
| Num layers | 32 | Depth of the model |
| Attention size | 9216 | 3072 × 3 (Q, K, V) |
| MLP expansion | 16384 | ~5x hidden size expansion |

In [ ]:
prompt = "The capital of France is"

# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

# Tokenize the input prompt
input_ids = input_ids.to("cuda")

# Get the output of the model before the lm_head
model_output = model.model(input_ids)

# Get the output of the lm_head
lm_head_output = model.lm_head(model_output[0])

In [ ]:
token_id = lm_head_output[0,-1].argmax(-1)
tokenizer.decode(token_id)

In [ ]:
model_output[0].shape

In [ ]:
lm_head_output.shape

## Understanding `lm_head_output[0, -1].argmax(-1)`

---

## What is `lm_head_output`?

After passing through the model, `lm_head_output` is a **3D tensor** with shape:
```
[batch_size, sequence_length, vocab_size]
     0             1                2
```

For example, if your prompt has 10 tokens and vocab size is 32064:
```
lm_head_output.shape = [1, 10, 32064]
```

Each position contains a **score for every possible next token** in the vocabulary.

---

## What is `[0, -1]`?
```python
lm_head_output[0, -1]
```

It's **numpy/tensor indexing**:

- `0` — selects the **first (and only) item in the batch**
- `-1` — selects the **last token position** in the sequence (Python's `-1` means last element)

We only care about the **last position** because that's where the model predicts the **next token** after your prompt.
```
lm_head_output[0, -1].shape = [32064]  ← one score per vocabulary token
```

---

## What is `.argmax(-1)`?
```python
.argmax(-1)
```

- `argmax` returns the **index of the highest value**
- `-1` means apply it along the **last dimension** (the vocab dimension of size 32064)
- So it finds which token ID has the **highest score** = the most likely next token
```
[0.1, 0.05, 0.9, 0.003, ...]  ←  32064 scores
       argmax → 2              ←  token ID 2 has highest score
```

---

## Full Picture
```python
token_id = lm_head_output[0, -1].argmax(-1)  # get ID of most likely next token
tokenizer.decode(token_id)                    # convert token ID back to text
```
```
lm_head_output         →  shape [1, 10, 32064]   (all token scores)
lm_head_output[0, -1]  →  shape [32064]           (scores at last position)
.argmax(-1)            →  single integer           (ID of best token)
tokenizer.decode(...)  →  "Hello" or "The" etc.   (human readable text)
```

This is essentially **greedy decoding** — always picking the single most probable next token.

In [ ]:
prompt = "Write a very long email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."

# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
input_ids = input_ids.to("cuda")

In [ ]:
%%timeit -n 1
# Generate the text
generation_output = model.generate(
  input_ids=input_ids,
  max_new_tokens=100,
  use_cache=True
)

In [ ]:
%%timeit -n 1
# Generate the text
generation_output = model.generate(
  input_ids=input_ids,
  max_new_tokens=100,
  use_cache=False
)